In [1]:
import pandas as pd
import numpy as np
import os
import json
import joblib
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier
import shap
import warnings
warnings.filterwarnings('ignore')

In [2]:
def load_all_patients(set_paths):
    dfs = []
    for folder in set_paths:
        files = os.listdir(folder)
        print(f"Loading {len(files)} files from {folder}...")
        for fname in files:
            if fname.endswith('.psv'):
                path = os.path.join(folder, fname)
                df = pd.read_csv(path, sep='|')
                df['patient_id'] = fname.replace('.psv', '')
                dfs.append(df)
    combined = pd.concat(dfs, ignore_index=True)
    print(f"\nTotal rows: {len(combined)}")
    print(f"Total patients: {combined['patient_id'].nunique()}")
    return combined

# Point these to your actual folders
SET_A = r'C:\Users\pradheepa jaya shree\Desktop\sepsis-dataset-1\training_setA'
SET_B = r'C:\Users\pradheepa jaya shree\Desktop\sepsis-dataset-1\training_setB'

data = load_all_patients([SET_A, SET_B])
data.head()

Loading 20336 files from C:\Users\pradheepa jaya shree\Desktop\sepsis-dataset-1\training_setA...
Loading 20000 files from C:\Users\pradheepa jaya shree\Desktop\sepsis-dataset-1\training_setB...

Total rows: 1552210
Total patients: 40336


,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,BaseExcess,HCO3,...,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel,patient_id
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,83.14,0,NaN,NaN,-0.03,1,0,p000001
1,97.0,95.0,NaN,98.0,75.33,NaN,19.0,NaN,NaN,NaN,...,NaN,NaN,83.14,0,NaN,NaN,-0.03,2,0,p000001
2,89.0,99.0,NaN,122.0,86.00,NaN,22.0,NaN,NaN,NaN,...,NaN,NaN,83.14,0,NaN,NaN,-0.03,3,0,p000001
3,90.0,95.0,NaN,NaN,NaN,NaN,30.0,NaN,24.0,NaN,...,NaN,NaN,83.14,0,NaN,NaN,-0.03,4,0,p000001
4,103.0,88.5,NaN,122.0,91.33,NaN,24.5,NaN,NaN,NaN,...,NaN,NaN,83.14,0,NaN,NaN,-0.03,5,0,p000001


In [3]:
label_counts = data['SepsisLabel'].value_counts()
print("SepsisLabel counts:")
print(label_counts)

neg = label_counts[0]
pos = label_counts[1]
scale_pos_weight = neg / pos

print(f"\nNon-sepsis rows: {neg}")
print(f"Sepsis rows:     {pos}")
print(f"Ratio:           {scale_pos_weight:.1f}x")
print(f"\nscale_pos_weight to use in XGBoost: {scale_pos_weight:.2f}")

SepsisLabel counts:
SepsisLabel
0    1524294
1      27916
Name: count, dtype: int64

Non-sepsis rows: 1524294
Sepsis rows:     27916
Ratio:           54.6x

scale_pos_weight to use in XGBoost: 54.60


In [4]:
def preprocess(df):
    df = df.copy()
    
    # ── 1. Forward fill within each patient, then backward fill ──
    df = df.groupby('patient_id', group_keys=False).apply(
        lambda x: x.ffill().bfill()
    )
    
    # ── 2. Fill anything still missing with column median ──
    df = df.fillna(df.median(numeric_only=True))
    
    # ── 3. Safety net ──
    df = df.fillna(0)
    
    # ── 4. Rolling mean (last 6 hours) per patient ──
    vitals = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'Resp']
    for col in vitals:
        df[f'{col}_6h_mean'] = (
            df.groupby('patient_id')[col]
            .transform(lambda x: x.rolling(6, min_periods=1).mean())
        )
        df[f'{col}_3h_mean'] = (
            df.groupby('patient_id')[col]
            .transform(lambda x: x.rolling(3, min_periods=1).mean())
        )
    
    # ── 5. Delta features (change from previous hour) ──
    for col in vitals:
        df[f'{col}_delta'] = df.groupby('patient_id')[col].diff().fillna(0)
    
    # ── 6. Ratio feature (shock index) ──
    df['shock_index'] = df['HR'] / (df['SBP'].replace(0, np.nan).fillna(1))
    
    return df

print("Running preprocessing... (this takes 2-4 minutes)")
data = preprocess(data)
print("Done!")
print(f"Total features now: {len(data.columns)}")

Running preprocessing... (this takes 2-4 minutes)
Done!
Total features now: 61


In [5]:
FEATURES = [
    # Raw vitals
    'HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'Resp',
    # Key labs
    'WBC', 'Creatinine', 'Lactate', 'Glucose',
    'Potassium', 'HCO3', 'pH', 'Platelets',
    # Demographics
    'Age', 'Gender', 'HospAdmTime', 'ICULOS',
    # Engineered features
    'HR_6h_mean', 'Resp_6h_mean', 'Temp_6h_mean',
    'SBP_6h_mean', 'O2Sat_6h_mean',
    'HR_3h_mean', 'Resp_3h_mean',
    'HR_delta', 'Resp_delta', 'Temp_delta', 'SBP_delta',
    'shock_index'
]

X = data[FEATURES]
y = data['SepsisLabel']
groups = data['patient_id']

# Split by patient — NOT by row
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
y_train = y.iloc[train_idx]
X_test  = X.iloc[test_idx]
y_test  = y.iloc[test_idx]

print(f"Train rows: {len(X_train)}")
print(f"Test rows:  {len(X_test)}")
print(f"Sepsis in train: {y_train.sum()}")
print(f"Sepsis in test:  {y_test.sum()}")

Train rows: 1241213
Test rows:  310997
Sepsis in train: 22669
Sepsis in test:  5247


In [6]:
print(f"Training with scale_pos_weight = {scale_pos_weight:.2f}")

model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',
    use_label_encoder=False,
    random_state=42,
    n_jobs=-1  # use all CPU cores
)

model.fit(X_train, y_train)
print("Training complete!")

Training with scale_pos_weight = 54.60
Training complete!


In [7]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))
print(f"AUROC: {roc_auc_score(y_test, y_proba):.4f}")
print("\nTarget: AUROC > 0.85")

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.87      0.93    305750
           1       0.07      0.61      0.13      5247

    accuracy                           0.86    310997
   macro avg       0.53      0.74      0.53    310997
weighted avg       0.98      0.86      0.91    310997

AUROC: 0.8271

Target: AUROC > 0.85


In [8]:
os.makedirs('models', exist_ok=True)

joblib.dump(model, 'models/sepsis_xgb_model.pkl')
json.dump(FEATURES, open('models/feature_columns.json', 'w'))

print("Saved:")
print("  models/sepsis_xgb_model.pkl")
print("  models/feature_columns.json")

Saved:
  models/sepsis_xgb_model.pkl
  models/feature_columns.json


In [9]:
explainer = shap.TreeExplainer(model)

# Test on one row
sample = X_test.iloc[[0]]
shap_vals = explainer.shap_values(sample)

# Top 6 features for that row
shap_series = pd.Series(shap_vals[0], index=FEATURES).sort_values(key=abs, ascending=False)
print("Top 6 contributing features for this prediction:")
print(shap_series.head(6))

Top 6 contributing features for this prediction:
Creatinine      0.609375
ICULOS         -0.418583
WBC            -0.208529
Potassium      -0.196945
MAP             0.186081
Temp_6h_mean   -0.181090
dtype: float32


In [10]:
# Cell 10 — Save population medians for API use
import json

medians = data[FEATURES].median().to_dict()
with open('models/population_medians.json', 'w') as f:
    json.dump(medians, f)

print("Saved models/population_medians.json")

Saved models/population_medians.json
